# Tutorial 4-2: Mitigating Algorithmic Bias
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

--- 
## Objective
In Tutorial 4-1, we learned how to audit a model to find bias. In this tutorial, we move to the next level: **intervening** to fix those biases. We will explore:
1. **Bias Mitigation via Sample Reweighing:** Mathematically adjusting the importance of training samples to counteract historical disparities.
2. **Solving Convergence Issues:** Using Feature Scaling to help traditional models like Logistic Regression settle on a solution.
3. **The Fairness-Accuracy Tradeoff:** Evaluating the model's overall performance after we force it to be more equitable.

## Part 1: Revisiting the Census Income Problem
We observed in our audit that the standard Logistic Regression model was significantly more likely to predict a 'High Income' for men than for women, even when other factors were equal. 

First, let's reload the data and fix the **convergence warning** using `StandardScaler`. Scaling ensures all features have a mean of 0 and a standard deviation of 1, allowing the optimization algorithm to converge quickly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

# Load the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = ['age', 'workclass', 'fnlwgt', 'education', 'edu_num', 'marital', 'occupation', 
           'relationship', 'race', 'sex', 'cap_gain', 'cap_loss', 'hours_per_week', 'country', 'income']
df = pd.read_csv(url, names=columns, skipinitialspace=True)

# Basic Prep
df['income'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)
X = pd.get_dummies(df.drop('income', axis=1))
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Standard Model (Biased)
model_standard = LogisticRegression(max_iter=1000)
model_standard.fit(X_train_scaled, y_train)
y_pred_std = model_standard.predict(X_test_scaled)

print(f"Standard Model Accuracy: {accuracy_score(y_test, y_pred_std):.4f}")

## Part 2: Bias Mitigation via Reweighing
We will implement a **Reweighing** strategy. The goal is to attach a weight to each sample during training. 

We calculate weights such that the privileged group (Men) and unprivileged group (Women) are balanced relative to the output label (Income). This forces the model to learn from 'counter-stereotypical' examples (like high-earning women) more heavily.

In [ ]:
def compute_rewriting_weights(df_train):
    # Calculate probabilities for reweighing
    n = len(df_train)
    n_pos = len(df_train[df_train['income'] == 1])
    n_neg = len(df_train[df_train['income'] == 0])
    
    weights = np.ones(n)
    
    for gender in ['Male', 'Female']:
        n_group = len(df_train[df_train['sex'] == gender])
        for label in [0, 1]:
            n_label = n_pos if label == 1 else n_neg
            n_group_label = len(df_train[(df_train['sex'] == gender) & (df_train['income'] == label)])
            
            # Expected count if gender/income were independent vs Actual observed count
            weight = (n_group * n_label) / (n * n_group_label)
            
            mask = (df_train['sex'] == gender) & (df_train['income'] == label)
            weights[mask] = weight
    return weights

# Compute weights using the training slice
train_df_slice = df.loc[X_train.index]
sample_weights = compute_rewriting_weights(train_df_slice)

# Train the 'Fair' Model
model_fair = LogisticRegression(max_iter=1000)
model_fair.fit(X_train_scaled, y_train, sample_weight=sample_weights)
y_pred_fair = model_fair.predict(X_test_scaled)

## 3. Comparative Performance Analysis
Does the mitigated model actually solve the bias? Let's look at the **Selection Rate** (how often the model predicts >$50k) for each gender across both models.

In [ ]:
test_df_slice = df.loc[X_test.index].copy()
test_df_slice['Standard_Pred'] = y_pred_std
test_df_slice['Fair_Pred'] = y_pred_fair

comparison = test_df_slice.groupby('sex')[['Standard_Pred', 'Fair_Pred']].mean()

plt.figure(figsize=(10, 6))
comparison.plot(kind='bar')
plt.title("Selection Rate (>50k Predictions) by Gender")
plt.ylabel("Percentage Predicted High Income")
plt.show()

print("Selection Rates:")
print(comparison)

print(f"\nStandard Model Accuracy: {accuracy_score(y_test, y_pred_std):.4f}")
print(f"Fair Model Accuracy: {accuracy_score(y_test, y_pred_fair):.4f}")

## Summary
By intervening in the training process, we have transformed the model's behavior:
1. **Fairness Gained:** The gap in high-income predictions between men and women has significantly closed.
2. **Technical Mastery:** We solved the convergence issue by adding **Feature Scaling**, which is mandatory for many traditional ML models.
3. **Tradeoff awareness:** Notice that the fair model's accuracy might be slightly lower than the standard model. This is the **Fairness-Accuracy Tradeoff**: we sacrifice a small amount of raw accuracy to prevent the model from learning and perpetuating harmful social biases.